# 02. TabPFN 베이스라인

- 전처리: `01` 베이스라인과 **똑같이 고정**
- 모델만 `RandomForestRegressor` → `TabPFNRegressor`로 교체
- 검증: 팀 공통 5-Fold (seed 42) · CV RMSE (원래 관객수 단위, 낮을수록 좋음)

## 0. 설치 (처음 한 번만)

In [ ]:
%pip install tabpfn

## 1. 라이브러리 · 데이터 로딩

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from sklearn import set_config
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tabpfn import TabPFNRegressor

# 한글 윈도우에서 모델 출력(HTML) 시 UnicodeDecodeError 나는 것 방지
set_config(display='text')

# 데이터 폴더 경로: 노트북과 같은 폴더면 '.' 그대로
DATA_DIR = '.'

# 팀 공통 설정
SEED = 42
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

train = pd.read_csv(os.path.join(DATA_DIR, 'movies_train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'movies_test.csv'))
submission = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))

## 2. 전처리 (01 베이스라인과 동일)

In [ ]:
def preprocess_baseline(train, test):
    train, test = train.copy(), test.copy()

    # 결측 많은 컬럼 · 감독명 · 제목 제거
    drop_cols = ['dir_prev_bfnum', 'director', 'title']
    train = train.drop(drop_cols, axis=1)
    test = test.drop(drop_cols, axis=1)

    # 상위 5개 배급사 외에는 '기타'
    top5 = train['distributor'].value_counts().index[:5]
    for df in (train, test):
        df['distributor'] = df['distributor'].where(df['distributor'].isin(top5), '기타')

    # 개봉일 → 년, 월
    for df in (train, test):
        date = pd.to_datetime(df['release_time'])
        df['년'] = date.dt.year
        df['월'] = date.dt.month
    train = train.drop(['release_time'], axis=1)
    test = test.drop(['release_time'], axis=1)

    # 원핫 인코딩
    train = pd.get_dummies(train)
    test = pd.get_dummies(test)

    X = train.drop(['box_off_num'], axis=1)
    y = train['box_off_num']
    X_test = test.reindex(columns=X.columns, fill_value=0)
    return X.astype(float), y, X_test.astype(float)

X, y, X_test = preprocess_baseline(train, test)
print(X.shape, X_test.shape)

## 3. 팀 공통 CV 함수

In [ ]:
def cv_rmse(make_model, X, y, log_target=False):
    """5-Fold CV RMSE (원래 관객수 단위).
    log_target=True 이면 log1p(y)로 학습하고 expm1로 되돌려서 RMSE 계산."""
    oof = np.zeros(len(y))
    for tr_idx, va_idx in kf.split(X):
        model = make_model()
        y_tr = np.log1p(y.iloc[tr_idx]) if log_target else y.iloc[tr_idx]
        model.fit(X.iloc[tr_idx], y_tr)
        pred = model.predict(X.iloc[va_idx])
        oof[va_idx] = np.expm1(pred) if log_target else pred
    rmse = mean_squared_error(y, oof) ** 0.5
    return rmse, oof

## 4. 비교 기준: RandomForest (01 베이스라인)

In [ ]:
rf_rmse, _ = cv_rmse(lambda: RandomForestRegressor(n_estimators=100, random_state=SEED), X, y)
print(f'RandomForest CV RMSE: {rf_rmse:,.0f}')

## 5. TabPFN

In [ ]:
make_tabpfn = lambda: TabPFNRegressor(device=DEVICE, random_state=SEED)

pfn_rmse, _ = cv_rmse(make_tabpfn, X, y)
print(f'TabPFN CV RMSE: {pfn_rmse:,.0f}')

## 6. (참고) 타깃 로그 변환 버전
RMSE는 대박 영화에 크게 좌우되므로, 로그 변환이 점수를 올릴지 떨어뜨릴지는 직접 비교해봐야 함.

In [ ]:
pfn_log_rmse, _ = cv_rmse(make_tabpfn, X, y, log_target=True)
print(f'TabPFN (log target) CV RMSE: {pfn_log_rmse:,.0f}')

## 7. 결과 비교

In [ ]:
result = pd.DataFrame({
    'model': ['RandomForest (01)', 'TabPFN', 'TabPFN (log target)'],
    'CV RMSE': [rf_rmse, pfn_rmse, pfn_log_rmse],
}).sort_values('CV RMSE')
result.style.format({'CV RMSE': '{:,.0f}'})

## 8. 전체 train으로 학습 후 제출 파일 생성
`USE_LOG`는 위 비교에서 더 좋았던 쪽으로 설정

In [ ]:
USE_LOG = pfn_log_rmse < pfn_rmse
print('로그 타깃 사용:', USE_LOG)

model = make_tabpfn()
model.fit(X, np.log1p(y) if USE_LOG else y)
pred = model.predict(X_test)
if USE_LOG:
    pred = np.expm1(pred)

submission['box_off_num'] = pred
submission.to_csv('02_tabpfn_baseline.csv', index=False)
submission.head()